In [3]:
from pathlib import Path
import sys

# Add the project root directory to sys.path
sys.path.append(str(Path().resolve().parent))

from src.processing.pde_ple import es  # Assuming Elasticsearch instance is here
from elasticsearch import helpers
from sentence_transformers import SentenceTransformer
import torch

# Parameters
chunk_index = "uc202-rex-chunks"
embedding_index = "uc202-rex-embeddings"
batch_size = 512
model_name = "intfloat/multilingual-e5-large-instruct"

# Load model
device = "cuda" if torch.cuda.is_available() else "cpu"
#model = SentenceTransformer(model_name, device=device)

# Create embedding index if not exists
if not es.indices.exists(index=embedding_index):
    es.indices.create(
        index=embedding_index,
        body={
            "mappings": {
                "properties": {
                    "embedding": {
                        "type": "dense_vector",
                        "dims": 1024,
                        "index": True,
                        "similarity": "cosine",
                    },
                    "chunk_id": {"type": "keyword"},
                    "original_doc_id": {"type": "keyword"},
                    "chunk_index": {"type": "integer"},
                    "chunk_content": {"type": "text"},
                }
            }
        },
    )

# Retrieve already embedded chunk_ids
print("Récupération des chunks déjà embarqués...")
existing_ids = set()

scroll = "15m"
query = {"query": {"match_all": {}}}

response = es.search(index=embedding_index, scroll=scroll, size=10000, body=query)
scroll_id = response["_scroll_id"]
hits = response["hits"]["hits"]

while hits:
    for doc in hits:
        existing_ids.add(doc["_source"]["chunk_id"])

    response = es.scroll(scroll_id=scroll_id, scroll=scroll)
    scroll_id = response["_scroll_id"]
    hits = response["hits"]["hits"]

print(f"Nombre de chunks déjà embarqués : {len(existing_ids)}")


/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/tmp/ipykernel_16264/1827356668.py:51: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  response = es.search(index=embedding_index, scroll=scroll, size=10000, body=query)
/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Récupération des chunks déjà embarqués...


/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-

Nombre de chunks déjà embarqués : 169343
